# Ebook AI – RAG test (Colab)
Self-contained test of the pipeline: PDFs in Supabase Storage -> Chroma index (local copy + Supabase copy, with `VERSION.json`) -> Gemini answers.

**Before running**, add these in Colab → 🔑 *Secrets* (enable notebook access):
`SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `GOOGLE_API_KEY` (optional: `SUPABASE_BUCKET`, default `ebook-ai`).

In [ ]:
!pip -q install supabase PyPDF2 langchain-core langchain-text-splitters langchain-chroma langchain-huggingface langchain-google-genai sentence-transformers

In [ ]:
import os, json, shutil, tempfile
from io import BytesIO
from datetime import datetime, timezone
from getpass import getpass

try:
    from google.colab import userdata
    def _secret(name, default=None):
        try: return userdata.get(name)
        except Exception: return os.environ.get(name, default)
except ImportError:
    def _secret(name, default=None): return os.environ.get(name, default)

for name in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "GOOGLE_API_KEY"):
    os.environ[name] = _secret(name) or getpass(f"{name}: ")
os.environ["SUPABASE_BUCKET"] = _secret("SUPABASE_BUCKET", "ebook-ai") or "ebook-ai"

BUCKET = os.environ["SUPABASE_BUCKET"]
LOCAL_DIR = "/content/chroma_db_local"
VERSION_FILE = "VERSION.json"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print("Config OK, bucket =", BUCKET)

## 1. Supabase Storage helpers (same logic as `supabase_storage.py`)

In [ ]:
from supabase import create_client
_k = os.environ["SUPABASE_SERVICE_KEY"]
assert not _k.startswith("sb_publishable_"), "SUPABASE_SERVICE_KEY is the PUBLISHABLE (anon) key - use the secret key (sb_secret_...) or legacy service_role key from Dashboard > Settings > API Keys"
sb = create_client(os.environ["SUPABASE_URL"], _k)

# Create the private bucket if it does not exist yet
if BUCKET not in [b.name for b in sb.storage.list_buckets()]:
    sb.storage.create_bucket(BUCKET, options={"public": False})
    print("Created bucket", BUCKET)
store = sb.storage.from_(BUCKET)

def list_files(prefix, page=100):
    prefix, out, offset = prefix.strip("/"), [], 0
    while True:
        items = store.list(prefix, {"limit": page, "offset": offset})
        for it in items:
            path = f"{prefix}/{it['name']}" if prefix else it["name"]
            out.extend(list_files(path) if it.get("id") is None else [path])
        if len(items) < page: break
        offset += page
    return out

def upload_file(path, local_path):
    with open(local_path, "rb") as f:
        store.upload(path, f.read(), {"content-type": "application/octet-stream", "upsert": "true"})

print("books/:", list_files("books"))

## 2. Upload PDFs to `books/` (skip if already there)

In [ ]:
from google.colab import files
up = files.upload()  # choose one or more .pdf files
for name, data in up.items():
    store.upload(f"books/{name}", data, {"content-type": "application/pdf", "upsert": "true"})
    print("uploaded books/" + name)

## 3. Build the index → local copy + Supabase copy

In [ ]:
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

documents = []
for name in list_files("books"):
    if not name.lower().endswith(".pdf"): continue
    try:
        reader = PdfReader(BytesIO(store.download(name)))
        text = " ".join((p.extract_text() or "") for p in reader.pages)
    except Exception as e:
        print("Skipping", name, e); continue
    if text.strip():
        documents += [Document(page_content=c, metadata={"file_name": name}) for c in splitter.split_text(text)]
print(len(documents), "chunks")
assert documents, "No PDF text found - upload PDFs first"

with tempfile.TemporaryDirectory() as tmp:
    vs = Chroma.from_documents(documents=documents, embedding=embeddings, persist_directory=tmp)
    try: vs._client._system.stop()
    except Exception: pass

    version = {
        "version": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
        "files": sorted({d.metadata["file_name"] for d in documents}),
        "chunks": len(documents),
        "embedding_model": MODEL_NAME,
    }
    json.dump(version, open(os.path.join(tmp, VERSION_FILE), "w"), indent=2)

    shutil.rmtree(LOCAL_DIR, ignore_errors=True)
    shutil.copytree(tmp, LOCAL_DIR)                       # local copy

    store.remove(list_files("chroma_db")) if list_files("chroma_db") else None
    for root, _, fs in os.walk(tmp):                      # Supabase copy
        for f in fs:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, tmp).replace(os.sep, "/")
            upload_file(f"chroma_db/{rel}", full)
print("Version:", version)
print("Supabase chroma_db/:", list_files("chroma_db"))

## 4. Verify: download the Supabase copy into a fresh folder and compare versions

In [ ]:
CHECK_DIR = "/content/chroma_db_from_supabase"
shutil.rmtree(CHECK_DIR, ignore_errors=True)
for name in list_files("chroma_db"):
    rel = name[len("chroma_db/"):]
    if any(p in ("", ".", "..") for p in rel.split("/")): continue   # path-traversal guard
    dest = os.path.join(CHECK_DIR, *rel.split("/"))
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    open(dest, "wb").write(store.download(name))

local_v  = json.load(open(os.path.join(LOCAL_DIR, VERSION_FILE)))["version"]
remote_v = json.load(open(os.path.join(CHECK_DIR, VERSION_FILE)))["version"]
print("local:", local_v, "| supabase:", remote_v, "| in sync:", local_v == remote_v)

## 5. Query with Gemini

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

vector_store = Chroma(embedding_function=embeddings, persist_directory=CHECK_DIR)  # use the Supabase copy
retriever = vector_store.as_retriever(search_type="similarity_score_threshold",
                                      search_kwargs={"k": 3, "score_threshold": 0.4})
prompt = PromptTemplate.from_template("""
You are a skilled assistant. Rewrite and summarize the context so it directly addresses the question.
If the data (context) doesn't exist, say so.

**Context:** {context}

**Question:** {input}

**Rewritten and Summarized Context:**
""")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

def ask(q):
    docs = retriever.invoke(q)
    ctx = "

".join(d.page_content for d in docs)
    print(f"[{len(docs)} chunks from {sorted({d.metadata['file_name'] for d in docs})}]")
    return (prompt | llm | StrOutputParser()).invoke({"context": ctx, "input": q})

print(ask("What is Flutter?"))